# (STM32F) Run for unmasked AES for STM32F3

In [ ]:
SCOPETYPE='OPENADC'
PLATFORM='CW308_STM32F3'
CRYPTO_TARGET='TINYAES128C'  #Change to TINYAES128C for TinyAES128
SS_VER='SS_VER_1_1'
board = 'stm32f3'

# (XMEGA) Run for unmasked AES for XMEGA

In [ ]:
SCOPETYPE='OPENADC'
PLATFORM='CWLITEXMEGA'
CRYPTO_TARGET='TINYAES128C' 
SS_VER='SS_VER_1_1'
board = 'xmega'

# Detect, Compile, and Flash to ChipWhisperer

In [ ]:
# Run setup script twice to correctly set capture clock
import os
setupScript = "/home/"+os.getenv("USER")+"/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"
%run "{setupScript}"
%run "{setupScript}"

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
#cd /home/dung-dang/chipwhisperer/firmware/mcu/simpleserial-aes
cd /home/$USER/chipwhisperer/firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 

In [ ]:
#cmd = "/home/" + os.getenv("USER") + "/chipwhisperer/firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM)
cmd = "/home/" + os.getenv("USER") + "/chipwhisperer/firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, cmd)
print(cmd)

# Set 'Offset' and 'Samples per Trace' parameters:

In [ ]:
import numpy as np
import time

ktp = cw.ktp.Basic()      # Set key and plaintext format
key, text = ktp.next()
target.set_key(key)

scope.adc.offset = 0     # 0 = start of program, increase value to record later into the program
scope.adc.samples = 5000  # 5000 is the max value for the ChipWhisperer Lite
ktp = cw.ktp.Basic()

In [ ]:
print(scope.clock)

# Key Setup

In [ ]:
dir(ktp)
initialKey = ktp.getInitialKey()
keyType = ktp.get_key_type()
print("Key Type: ", keyType)

# The appropriate key can be uncommented based on the sspecific dataset,
# Note that the default key is automatically set: We refer to the default key as K1
# Different key options that we use in data collection should be uncommented one at a time:

key_str = 'aa,80,d8,a7,84,d3,3f,5c,0b,90,a9,85,20,8e,ff,4a'  # K2
# key_str = 'd2,d5,01,68,82,83,91,43,96,9e,e9,a2,53,a7,52,e1'  # K3
# key_str = 'e6,de,35,a9,a5,23,19,df,c6,cc,bb,ba,c1,36,c3,bf'  # K4
# key_kareem = '2b,7e,15,16,28,ae,d2,a6,ab,f7,15,88,09,cf,4f,3c'
ktp.fixed_key = False
ktp.setInitialKey(key_str)

ktp.fixed_key = True
ktp.fixed_text = False
key, text = ktp.next()
print("Key Utilized: ", key)

In [ ]:
target.set_key(key)
for i in range(4):
    key, text = ktp.next()
    print(key)
    print(text)
    print('==============')

# Collect A Small Number of Sample Traces for Initial Testing

In [ ]:
from tqdm import trange
trace_array = []
textin_array = []
textout_array = []

# Set N as a small number, e.g., 10 or 20 
N = 15

for i in trange(N, desc='Capturing traces'):
    scope.arm()
    target.simpleserial_write('p', text)
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    response = target.simpleserial_read('r', 16)
    
    # Record trace, plaintext, and ciphertext to confirm successful encryption
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    textout_array.append(response)
    
    key, text = ktp.next() 

In [ ]:
print(f"Number of samples for whole implementation:{scope.adc.trig_count}")

# Visualize the Shape of A Collected Power Trace:

In [ ]:
%matplotlib inline
import matplotlib.pylab as plt
from matplotlib.pyplot import MultipleLocator

def annotate_zone(ax, start, end, text, color="red", h_offset=0.0):
    ymin, ymax = ax.get_ylim()
    y_arrow = ymin + (ymax - ymin) * (0.75 + h_offset) 
    
    ax.axvspan(start, end, color=color, alpha=0.15) 
    ax.annotate('', xy=(start, y_arrow), xytext=(end, y_arrow), arrowprops=dict(arrowstyle="<->", color=color, lw=2))
    ax.text((start + end)/2, y_arrow + (ymax-ymin)*0.02, text, color=color, ha="center", va="bottom", fontweight="bold")

offset = 0
startPOI = 0 
endPOI = 5000 

plt.figure(figsize=(13,8))
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(1000))

# choose the first trace from sample collection to generate a plot
for i in range(1):  
    y_data = trace_array[i][startPOI:endPOI]
    plt.plot(range(offset, offset + len(y_data)), y_data, alpha=0.7)

plt.xlabel("Timestamp")
plt.ylabel("Normalized Voltage Drop")
#plt.xlim(1200, 2100)


plt.ylim(min(y_data)*1.1, max(y_data)*1.1) 

# Highlight
#annotate_zone(ax, 888, 18208, "Mask Gen", color="C3")
# annotate_zone(ax, 1244, 2012, "SubBytes r-1", color="C2", h_offset=0.15) 
#annotate_zone(ax, 3936, 4720, "SubBytes r-2", color="C2", h_offset=0.15) 
#annotate_zone(ax, 25096, 25960, "SubBytes r-3", color="C2", h_offset=0.15) 
#annotate_zone(ax, 1000, 2499, "100 no-ops", color="C1", h_offset=-0.15) 
#annotate_zone(ax, 21301, 26000, "300 no-ops", color="C1", h_offset=-0.15)
#annotate_zone(ax, 27200, 28800, "100 no-ops", color="C1", h_offset=-0.15)
#annotate_zone(ax, 30000, 34500, "300 no-ops", color="C1", h_offset=-0.15)

plt.grid(True, alpha=0.3)
#plt.tight_layout()
plt.savefig(r"/home/" + os.getenv("USER") + "/Documents/Trace1.jpg")
plt.show()


In [ ]:
print(f"Number of samples for whole implementation:{scope.adc.trig_count}")

# Verify Ciphertexts Were Calculated Correctly:

In [ ]:
# Requires "pip install PyCryptodomex"
from Cryptodome.Cipher import AES
from Cryptodome.Util.Padding import pad, unpad

def aesECBEnc(key: bytes, message: bytes) -> bytes:
    """aesECBEnc takes a key and message and encrypts the message using AES-256 in ECB mode.
    :param key: A 256-bit key to use for the encryption
    :param message: The message to encrypt, in bytes
    :return: A bytes object of the encrypted ciphertext"""
    cipher = AES.new(key, AES.MODE_ECB)
    encryptedData = cipher.encrypt(message)  # Need to add padding as recommended by library
    return encryptedData

def aesECBDec(key: bytes, ciphertext: bytes) -> bytes:
    """aesECBDec takes a key and ciphertext and decrypts the ciphertext using AES-256 in ECB mode.
    :param key: The same 256-bit key used for encryption
    :param ciphertext: The ciphertext to decrypt, in bytes
    :return: The original encrypted message, in bytes"""
    cipher = AES.new(key, AES.MODE_ECB)
    decryptedData = unpad(cipher.decrypt(ciphertext), AES.block_size)  # Now need to remove padding if added
    return decryptedData
testsPassed = True
for i in range(len(trace_array)):
    print("Key:       ",bytes.hex(bytes(key)))
    print("Plaintext: ",bytes.hex(bytes(textin_array[i])))
    print("Ciphertext:",bytes.hex(bytes(textout_array[i])))
    ciphertext = aesECBEnc(bytes(key), bytes(textin_array[i]))
    calculatedCiphertext = bytes.hex(ciphertext)
    if bytes.hex(bytes(textout_array[i])) == calculatedCiphertext:
        print(f"            {calculatedCiphertext} <--- Test {i+1} passed")
    else:
        print(f"            {calculatedCiphertext} <--- Test {i+1} failed")
        testsPassed = False
    print("")

# Collect N Power Traces for Side-Channel Analysis 

In [ ]:
import time
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.*")
trace_array = []
textin_array = []
collectionStart = time.time()

# N is the number of traces we would like to collect, 
# For TinyAES (i.e., unmasked AES), typically 5,000 traces would be sufficient to recover key bytes.  
N = 5000 

for i in trange(N, desc='Capturing traces'):
    scope.arm()
    target.simpleserial_write('p', text)
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    response = target.simpleserial_read('r', 16)
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    key, text = ktp.next()
         
collectionEnd = time.time()
collectionDuration = collectionEnd - collectionStart

# TVLA Functions

In [ ]:
import time
from datetime import datetime
import os
import sys
from math import sqrt
from operator import itemgetter
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,  # 0
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,  # 1
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,  # 2
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,  # 3
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,  # 4
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,  # 5
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,  # 6
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,  # 7
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,  # 8
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,  # 9
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,  # a
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,  # b
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,  # c
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,  # d
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,  # e
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16  # f
]


def data_info(data):
    """
    This function prints the information about the dataset.
    :param data: Loaded NPZ file
    :return: None
    """
    # loading the dataset
    powerTraces, plainText, key = data['power_trace'], data['plain_text'], data['key']
    print('Shape of the power traces: ', powerTraces.shape)
    print('Shape of the plaintext   : ', plainText.shape)
    print('Ground Truth for the key : ', key)


def aesInternal(inp_data_byte, key_byte):
    """
    This function performs XOR operation between the input byte and key byte which is used as label.
    :param inp_data_byte: Input data (plaintext) byte
    :param key_byte: Key byte
    """
    inp_data_byte = inp_data_byte.astype(np.uint8)
    key_byte = key_byte.astype(np.uint8)
    return sbox[inp_data_byte ^ key_byte]


def allkeys(h5File):
    """Recursively find all keys in an h5py.Group."""
    keys = (h5File.name,)
    if isinstance(h5File, h5py.Group):
        for key, value in h5File.items():
            if isinstance(value, h5py.Group):
                keys = keys + allkeys(value)
            else:
                keys = keys + (value.name,)
    return keys


def loadDataTVLA(targetByte: int, startPOI: int, endPOI: int, fileName: str, numberOfTraces: int):
    """
    This function loads the specified file and generates features and labels for the dataset.
    :param targetByte: Target byte
    :param startPOI: Start point of interest window
    :param endPOI: End point of interest window
    :param fileName: Path of the file to load
    :param numberOfTraces: Number of traces to load
    """
    try:
        train_data_whole_pack = np.load(fileName)
        data_info(train_data_whole_pack)
    except OSError:
        print(f"[FATAL] -- Could not access {fileName}")
        sys.exit()

    powerTraces, traceLabels, keyByte = genFeaturesAndLabels256TVLA(train_data_whole_pack, targetByte, startPOI, endPOI)
    powerTraces = powerTraces[:numberOfTraces, :]
    traceLabels = traceLabels[:numberOfTraces]
    return powerTraces, traceLabels, keyByte


def loadDataTVLACHES(targetByte: int, startPOI: int, endPOI: int, fileName: str, numberOfTraces: int):
    with h5py.File(fileName) as in_file:
        print(allkeys(in_file))
        return np.array(in_file['Profiling_traces/traces'][:, startPOI:endPOI]), \
            np.array(in_file['Profiling_traces/metadata'][:numberOfTraces]['labels']), \
            np.array(in_file['Profiling_traces/metadata'][:numberOfTraces]['key'][:, targetByte])


def genFeaturesAndLabels256TVLA(data, input_target_byte, startPOI, endPOI):
    """
    Generates features and labels for the passed NPZ dataset.
    :param data: Loaded NPZ file
    :param input_target_byte: Target byte
    :param startPOI: Start index of the power traces
    :param endPOI: End index of the power traces
    """
    powerTraces, plainText, key = data['power_trace'], data['plain_text'], data['key']
    keyByte = key[input_target_byte]

    traceLabels = []
    for i in range(plainText.shape[0]):
        text_i = plainText[i]
        label = aesInternal(text_i[input_target_byte], keyByte)  # key[i][input_key_byte]
        traceLabels.append(label)

    traceLabels = np.array(traceLabels)
    if not isinstance(powerTraces, np.ndarray):
        powerTraces = np.array(powerTraces)
    powerTraces = powerTraces[:, startPOI:endPOI]

    return powerTraces, traceLabels, keyByte


def calculateTVLAValues(arr):
    """
    This function calculates the mean, variance, and length of the array.
    :param arr: Numpy array
    :return: Returns the mean, variance, and length of the array.
    """
    average = np.average(arr)
    if np.isnan(average):
        average = 0
    var = np.var(arr, ddof=1)
    if np.isnan(var):
        var = 0
    length = len(arr)
    return average, var, length


def exportTVLAToCSV(tValues, startPOI, endPOI, tByte, outputDir):
    """
    This function saves the test vector leakage assessment results to a CSV file.
    :param tValues: T-values from TVLA
    :param startPOI: Start point of interest window
    :param endPOI: End point of interest window
    :param tByte: Target byte
    :param outputDir: Output directory for the CSV file
    :return: None
    """
    saveFileName = "TVLAResultsCSV" + str(tByte) + datetime.today().strftime("%Y.%m.%d_%H.%M.%S") + '.csv'
    exportFilePath = os.path.join(outputDir, saveFileName)

    data = zip(range(startPOI + 1, endPOI + 1), tValues)  # X: 0 to n, Y: t-values from TVLA
    tvlaDataFrame = pd.DataFrame(data)
    tvlaDataFrame.to_csv(exportFilePath, index=False, header=["Time", "T-Values"])
    print(f"Test vector leakage assessment results sucessfully saved to csv file: {exportFilePath}")


def print_testing_test_vector_leakage_assessment_results(mean, variance, size, list_name, name):
    """
    This function prints the test vector leakage assessment intermediate values.
    """
    print(f"For set: {name}")
    print(f"Set Elements: {list_name}")
    print(f"Mean: {mean}")
    print(f"Variance: {variance}")
    print(f"Size: {size}")


def compute_test_vector_leakage_assessment(powerTraces, traceLabels, keyByte, debug=False):
    """
    This function computes the test vector leakage assessment given the power traces, labels, and key byte value.
    :param powerTraces: Numpy array of power traces
    :param traceLabels: Generated labels for the power traces
    :param keyByte: Key byte value
    :param debug: Prints additional information if enabled
    :return:
    """
    t_vals = []
    for j in range(np.shape(powerTraces)[1]):  # Each column (time sample) of the power_traces array is analyzed.
        curr_power_traces_col = powerTraces[:, j]
        Q0 = []  # Lists Q_0 and Q_1 are created.
        Q1 = []
        for k in range(np.shape(curr_power_traces_col)[0]):  # Each row of the current power_traces column is analyzed.
            # If the traceLabel in a trace is equal to the key byte, put it in the fixed set and the rest in the random set
            # When the intermediate value of encryption (after sbox) is equal to the key
            if traceLabels[k] == keyByte:
                Q0.append(curr_power_traces_col[k])
            else:
                Q1.append(curr_power_traces_col[k])

        # Calculate mean, variance, and length of both sets
        u0, v0, n0 = calculateTVLAValues(np.array(Q0))  # average, var, length
        u1, v1, n1 = calculateTVLAValues(np.array(Q1))
        denominator = (sqrt(((v0 / n0) if n0 != 0 else 0) + ((v1 / n1) if n1 != 0 else 0)))
        if denominator != 0:
            t = (u0 - u1) / denominator  # The t value is calculated and appended to a list of t_vals.
        else:
            t = 0
        t_vals.append(t)  # This list contains t_vals for every time sample.
        if debug:  # If debug is enabled, additional information will be printed to the screen.
            print("Round {}".format(j + 1))
            print_testing_test_vector_leakage_assessment_results(u0, v0, n0, Q0, "Q0")
            print_testing_test_vector_leakage_assessment_results(u1, v1, n1, Q1, "Q1")
            print("\tThe test vector leakage result is: {}".format(t))
    print(np.shape(powerTraces)[1])
    print(np.shape(curr_power_traces_col)[0])
    return t_vals

def tvlaGraph(tValues, startPOI, endPOI, targetByte, stepWidth, savePath, show=False):
    """
    This function generates a graph of the test vector leakage assessment results.
    :param tValues: List of t-values
    :param startPOI: Start point of interest window
    :param endPOI: End point of interest window
    :param targetByte: Target byte
    :param stepWidth: Step width for the x-axis
    :param datasetStr: Name of the dataset
    :param savePath: Output directory for the graph
    :return: None
    """

    # Big font
    # plt.rcParams.update({'font.size': 26})
    # plt.rc('legend', fontsize=26)
    print(len(tValues))
    plt.plot(np.arange(startPOI, endPOI), np.abs(tValues), label="TVLA", color='b')
    #plt.tight_layout(pad=4)
    plt.title(f"TVLA Byte {targetByte}")
    plt.xlabel("Timestamp")
    plt.ylabel("T-Values")
    plt.axhline(y=4.5, color='r', linestyle='--')
    # plt.axhline(y=-4.5, color='r', linestyle='--')
    # Set legend for 4.5 line
    plt.legend(["T-Values", '+/- 4.5'], loc="upper left", framealpha=1)

    # Set the x ticks and labels
#     x_ticks = np.arange(startPOI, endPOI, stepWidth)
#     plt.xticks(x_ticks, np.round(x_ticks, 0).astype(int))
    figure = plt.gcf()

    plt.margins(x=0)  # Tight plot bounds (no starting and ending whitespace)
    xTicks = np.arange(poiStart,poiEnd+stepWidth, stepWidth)
    plt.xticks(xTicks, xTicks)
    plt.xlim(poiStart, poiEnd)
    if show:
        plt.show()
    else:
        plt.savefig(savePath)
    # plt.clf()
    
def plot3Activities(powerTraces, plainText, key, input_target_byte, startPOI, endPOI, stepSize, savePath):
    keyByte = key[input_target_byte]  #get 1 byte from the key

    traceLabels = []
    for i in range(plainText.shape[0]):
        text_i = plainText[i]
        label = aesInternal(text_i[input_target_byte], keyByte)  # key[i][input_key_byte]
        traceLabels.append(label)

    traceLabels = np.array(traceLabels)
    if not isinstance(powerTraces, np.ndarray):
        powerTraces = np.array(powerTraces)
    powerTraces = powerTraces[:, startPOI:endPOI]
    tvlaValues = compute_test_vector_leakage_assessment(powerTraces, traceLabels, keyByte, debug=False)
    tvlaGraph(tvlaValues, startPOI, endPOI, input_target_byte, stepSize, savePath)

# CPA Functions

In [ ]:
import time
from datetime import datetime
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import sklearn

sbox = [
    # 0    1     2     3     4     5     6     7     8     9     a     b     c     d     e     f
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,  # 0
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,  # 1
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,  # 2
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,  # 3
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,  # 4
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,  # 5
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,  # 6
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,  # 7
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,  # 8
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,  # 9
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,  # a
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,  # b
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,  # c
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,  # d
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,  # e
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16,  # f
]

inv_sbox = [
    # 0    1     2     3     4     5     6     7     8     9     a     b     c     d     e     f
    0x52, 0x09, 0x6a, 0xd5, 0x30, 0x36, 0xa5, 0x38, 0xbf, 0x40, 0xa3, 0x9e, 0x81, 0xf3, 0xd7, 0xfb,  # 0
    0x7c, 0xe3, 0x39, 0x82, 0x9b, 0x2f, 0xff, 0x87, 0x34, 0x8e, 0x43, 0x44, 0xc4, 0xde, 0xe9, 0xcb,  # 1
    0x54, 0x7b, 0x94, 0x32, 0xa6, 0xc2, 0x23, 0x3d, 0xee, 0x4c, 0x95, 0x0b, 0x42, 0xfa, 0xc3, 0x4e,  # 2
    0x08, 0x2e, 0xa1, 0x66, 0x28, 0xd9, 0x24, 0xb2, 0x76, 0x5b, 0xa2, 0x49, 0x6d, 0x8b, 0xd1, 0x25,  # 3
    0x72, 0xf8, 0xf6, 0x64, 0x86, 0x68, 0x98, 0x16, 0xd4, 0xa4, 0x5c, 0xcc, 0x5d, 0x65, 0xb6, 0x92,  # 4
    0x6c, 0x70, 0x48, 0x50, 0xfd, 0xed, 0xb9, 0xda, 0x5e, 0x15, 0x46, 0x57, 0xa7, 0x8d, 0x9d, 0x84,  # 5
    0x90, 0xd8, 0xab, 0x00, 0x8c, 0xbc, 0xd3, 0x0a, 0xf7, 0xe4, 0x58, 0x05, 0xb8, 0xb3, 0x45, 0x06,  # 6
    0xd0, 0x2c, 0x1e, 0x8f, 0xca, 0x3f, 0x0f, 0x02, 0xc1, 0xaf, 0xbd, 0x03, 0x01, 0x13, 0x8a, 0x6b,  # 7
    0x3a, 0x91, 0x11, 0x41, 0x4f, 0x67, 0xdc, 0xea, 0x97, 0xf2, 0xcf, 0xce, 0xf0, 0xb4, 0xe6, 0x73,  # 8
    0x96, 0xac, 0x74, 0x22, 0xe7, 0xad, 0x35, 0x85, 0xe2, 0xf9, 0x37, 0xe8, 0x1c, 0x75, 0xdf, 0x6e,  # 9
    0x47, 0xf1, 0x1a, 0x71, 0x1d, 0x29, 0xc5, 0x89, 0x6f, 0xb7, 0x62, 0x0e, 0xaa, 0x18, 0xbe, 0x1b,  # a
    0xfc, 0x56, 0x3e, 0x4b, 0xc6, 0xd2, 0x79, 0x20, 0x9a, 0xdb, 0xc0, 0xfe, 0x78, 0xcd, 0x5a, 0xf4,  # b
    0x1f, 0xdd, 0xa8, 0x33, 0x88, 0x07, 0xc7, 0x31, 0xb1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xec, 0x5f,  # c
    0x60, 0x51, 0x7f, 0xa9, 0x19, 0xb5, 0x4a, 0x0d, 0x2d, 0xe5, 0x7a, 0x9f, 0x93, 0xc9, 0x9c, 0xef,  # d
    0xa0, 0xe0, 0x3b, 0x4d, 0xae, 0x2a, 0xf5, 0xb0, 0xc8, 0xeb, 0xbb, 0x3c, 0x83, 0x53, 0x99, 0x61,  # e
    0x17, 0x2b, 0x04, 0x7e, 0xba, 0x77, 0xd6, 0x26, 0xe1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0c, 0x7d,  # f
]

def dataInfo(data):
    """
    This function prints the information of the dataset.
    """
    power_traces, plain_text, key = data['power_trace'], data['plain_text'], data['key']
    print(f'index of data {data.files}')
    print('shape of the traces: ', power_traces.shape)
    print('shape of the plaintext : ', plain_text.shape)
    print('Ground Truth for the key : ', key)


def aesInternal(inp_data_byte, key_byte):
    """
    This function performs XOR operation between the input byte and key byte which is used as label.
    """
    inp_data_byte = inp_data_byte.astype(np.uint8)
    key_byte = np.uint8(key_byte)
    return sbox[inp_data_byte ^ key_byte]


def calc_hamming_weight(n):
    return bin(n).count("1")


def std_dev(X, X_bar):
    mean_x = X - X_bar
    x_square = []
    for i in range(X.shape[0]):
        tmp = mean_x[i, :]
        tmp_2 = np.square(tmp)
        x_square.append(tmp_2)

    x_square = np.array(x_square)
    sum_x_square = np.sum(x_square, axis=0)
    x_sqrt = np.sqrt(sum_x_square)
    return x_sqrt


def cov(X, X_bar, Y, Y_bar):
    mean_x = X - X_bar
    mean_y = Y - Y_bar
    product_x_y = mean_x * mean_y
    sum_res = np.sum(product_x_y, axis=0)
    return sum_res


def createNonProfilingGraph(distinguisherValues, xLabel, yLabel, title, maxMin: str, savePath, show=False,
                            correctKey=None):
    """
    Create a graph of the given distinguisher values to demonstrate the success of the non-profiling attack
    :param distinguisherValues: Accuracy or loss values for each byte
    :param xLabel: Label for the x-axis
    :param yLabel: Label for the y-axis
    :param title: Title for the graph
    :param maxMin: Should be 'max' or 'min' to indicate whether the max or min value should be highlighted
    :param savePath: The full path, including the file name, to save the graph to
    :return: None
    """

    # Create a line graph of given distinguisher values
    x_values = list(range(len(distinguisherValues)))
    plt.plot(x_values, distinguisherValues)

    # Create horizontal line for max/min value and make legend containing that value
    if maxMin.lower() == 'min':
        plt.axhline(y=min(distinguisherValues), color='r', linestyle='--')
        if correctKey is not None:
            plt.legend([f'Min value: {min(distinguisherValues):.4f} at byte: {str(distinguisherValues.index(min(distinguisherValues)))}\nCorrect Key: {correctKey}'])
        else:
            plt.legend([f'Min value: {min(distinguisherValues):.4f} at byte: {str(distinguisherValues.index(min(distinguisherValues)))}'])
    elif maxMin.lower() == 'max':
        plt.axhline(y=max(distinguisherValues), color='g', linestyle='--')
        if correctKey is not None:
            plt.legend([f'Max value: {max(distinguisherValues):.4f} at byte: {str(distinguisherValues.index(max(distinguisherValues)))}\nCorrect Key: {correctKey}'])
        else:
            plt.legend([f'Max value: {max(distinguisherValues):.4f} at byte: {str(distinguisherValues.index(max(distinguisherValues)))}'])

    # Set the labels for the axes
    plt.title(title)
    plt.xlabel(xLabel)
    plt.ylabel(yLabel)
    if show:
        plt.show()
    else:
        plt.savefig(savePath)
    # plt.clf()
    
def plot4Activities(traces, plaintexts, key, targetByte, startPOIWindow, endPOIWindow, savePath, show=False):
    key_byte_value = key[targetByte]
    traces = np.array(traces)[:, startPOIWindow:endPOIWindow]

    # Calculate hamming weights
    HW = []
    for i in range(0, 256):
        hw_val = calc_hamming_weight(i)
        HW.append(hw_val)

    t_bar = np.mean(traces, axis=0)
    o_t = std_dev(traces, t_bar)
    cpaValues = [0] * 256
    for keyGuess in tqdm(range(256)):
        hammingWeightLabels = np.array([[HW[aesInternal(plaintext[targetByte], keyGuess)] for plaintext in plaintexts]]).transpose()
        hws_bar = np.mean(hammingWeightLabels, axis=0)
        o_hws = std_dev(hammingWeightLabels, hws_bar)
        correlation = cov(traces, t_bar, hammingWeightLabels, hws_bar)
        stdDevMult = (o_t * o_hws)
        stdDevMult = np.where(stdDevMult == 0, 0.0000001, stdDevMult)  # Workaround for synthetic traces
        cpaOutput = correlation / stdDevMult
        cpaValues[keyGuess] = max(abs(cpaOutput))

    guess = np.argmax(cpaValues)
    guess_corr = max(cpaValues)
    sortedCpaValues = np.sort(cpaValues)
    createNonProfilingGraph(cpaValues, "Key Guess", "Acc.", f"CPA Byte {targetByte}, POI: {startPOIWindow}-{endPOIWindow}",
                            "max", savePath, show=show, correctKey=key_byte_value)
    # plt.clf()

# Run TVLA and CPA on collected traces

In [ ]:
%matplotlib inline
#Convert key, trace and plaintext
keyNPArray = np.array(key)  
trace_array = np.array(trace_array)
textin_array = np.array(textin_array)
poiStart = 0
poiEnd = 5000

# choose a key byte index to attack, any index from 0 to 15
target_byte = 2

# TVLA Analysis
# plot3Activities(trace_array, textin_array, keyNPArray, target_byte, poiStart, poiEnd, 500, r"/home/" + os.getenv("USER") + "/Documents/TVLA-Graph1.jpg")

# CPA Analysis 
plot4Activities(trace_array, textin_array, keyNPArray, target_byte, poiStart, poiEnd, r"/home/" + os.getenv("USER") + "/Documents/CPA-Graph1.jpg")



# Save Collected Traces:


In [ ]:
from datetime import datetime
# Save traces under the following filename, naming should follow these conventions:
# <Device Type Indicator Letter><Device Number>_K<Key Number>_<Number of traces in thousands>.npz
datasetName='STM32_K2_5k_TinyAES_20260802-001'
datasetExtension = ".npz"
output_path="/home/boyang/Documents/"+datasetName  # Save to Ubuntu Desktop
os.makedirs(output_path, exist_ok=True)

outpath = os.path.join(output_path, datasetName+datasetExtension)
np.savez(outpath, power_trace=trace_array, plain_text=textin_array, key=key)  # NPZ Format

# Export Metadata:

In [ ]:
# Export collection metadata to a text document
import hashlib
exportLogSavePath = os.path.join(output_path, "reproducibilityLog.log")
exportLog = list()
exportLog.append(f"##########################################################################")
exportLog.append(f"### Collection finished at {datetime.today().strftime('%m_%d_%Y %H:%M:%S')} by {os.getenv('USER')}")
exportLog.append(f"### Collection process took {collectionDuration:.2f}s.")
exportLog.append(f"##########################################################################")
exportLog.append(f"Target Parameters: ")
exportLog.append(f" * Target Board: {board}")
exportLog.append(f" * Target Implementation: {CRYPTO_TARGET}")
exportLog.append(f" * Target Platform: {PLATFORM}")
exportLog.append(f" * Simpleserial Version: {SS_VER}")
exportLog.append(f" * Target programmer: {prog}")
exportLog.append(f" * Target bitstream file: {cmd}")
exportLog.append(f"")
exportLog.append(f"Scope Parameters: ")
exportLog.append(f" * Scope offset: {scope.adc.offset}")
exportLog.append(f" * Scope samples per trace: {scope.adc.samples}")
exportLog.append(f" * Scope mode: {scope.gain.mode} -> Scope basic_mode: {scope.adc.basic_mode}")
exportLog.append(f" * Scope Gain Level: {scope.gain.gain} -> Gain Db: {scope.gain.db}")
exportLog.append(f" * Scope trig_count: {scope.adc.trig_count}")
exportLog.append(f" * Scope clkgen_div: {scope.clock.clkgen_div}")
exportLog.append(f" * Scope clkgen_freq: {scope.clock.clkgen_freq} -> {scope.clock.clkgen_freq/10e5:.3f} MHz")
exportLog.append(f" * Scope adc_src: {scope.clock.adc_src}")
exportLog.append(f" * Scope adc_freq: {scope.clock.adc_freq} -> {scope.clock.adc_freq/10e5:.3f} MHz")
exportLog.append(f" * Scope adc_rate: {scope.clock.adc_rate}")
exportLog.append(f"")
exportLog.append(f"Trace Collection Parameters: ")
exportLog.append(f" * Fixed key used? : {ktp.fixed_key}")
exportLog.append(f" * Tests Passed?: True") if testsPassed else exportLog.append(f" * Tests Passed?: FALSE <---")
exportLog.append(f" * Utilized Key: {initialKey}") if ktp.fixed_key else exportLog.append(f" * Dynamic Key used!")
exportLog.append(f" * Number of traces collected: {N}")
exportLog.append(f"")
exportLog.append(f"Dataset Parameters: ")
exportLog.append(f" * Saved dataset name: {datasetName}")
exportLog.append(f" * Saved dataset path: {outpath}")
exportLog.append(f" * Saved dataset size: {(os.path.getsize(outpath)/(1024*1024)):.3f} MBs")
exportLog.append(f" * Saved dataset hash (SHA256): {hashlib.sha256(open(outpath, 'rb').read()).hexdigest()}")
exportLog.append(f"")
with open(exportLogSavePath, 'w') as f:
    for line in exportLog:
        f.write(f"{line}\n")
plt.clf()
x_locator = MultipleLocator(2000)
ax = plt.gca()
ax.xaxis.set_major_locator(x_locator)
plt.xlabel("Timestamp")
plt.ylabel("Normalized Voltage Drop")
plt.plot(trace_array[2][0:5000], color='r')
plt.savefig(os.path.join(output_path, "traceFigure.png"))

# Also create a PDF of the report
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.units import inch

def setPDFMetadata(canvas, doc):
    canvas.setTitle("Reproducibility Report")
    canvas.setSubject("Side-channel Analysis Dataset Documentation")

pdf_path = os.path.join(output_path, "reproducibility_report.pdf")
doc = SimpleDocTemplate(
    pdf_path,
    pagesize=letter,
    leftMargin=1*inch,  # 1 inch margins
    rightMargin=1*inch,
    topMargin=1*inch,
    bottomMargin=1*inch
)

# Custom styles
styles = getSampleStyleSheet()
style_normal = styles['Normal']
style_normal.fontName = 'Times-Roman'
style_normal.fontSize = 12
style_normal.leading = 14

style_header = ParagraphStyle(
    'Header',
    parent=style_normal,
    fontSize=14,
    spaceAfter=20,
    alignment=1   # Center alignment
)

# Add log content to elements
elements = []
for line in exportLog:
    elements.append(Paragraph(line.replace(" * ", "&nbsp;&nbsp;* "), style_normal))

# Put image on a new page
elements.append(PageBreak())
image_path = os.path.join(output_path, "traceFigure.png")
elements.append(Paragraph(f"Example Trace from {datasetName} dataset:", style_header))

# Load and scale image
img = Image(image_path)
img_width = 6 * inch  # Set desired width
scaling_factor = img_width / img.drawWidth
img.drawWidth = img_width
img.drawHeight = img.drawHeight * scaling_factor
img_table = Table([[img]], colWidths=doc.width)
img_table.setStyle(TableStyle([
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE')
]))
elements.append(img_table)

doc.build(elements, onFirstPage=setPDFMetadata)  # Build PDF and export

# Close-out ChipWhisperer connection
scope.dis()
target.dis()

# Read the data from a .npz file

In [ ]:
import numpy as np
data = np.load("/home/boyang/Documents/STM32_K3_5k_TinyAES_20260802/STM32_K3_5k_TinyAES_20260802.npz")
trace_array = data['power_trace']
textin_array = data['plain_text']
key = data['key']

#print(data['key'])
print(key)
print(trace_array)
print(textin_array)
print(trace_array.shape)
print(textin_array.shape)

data.close()